In [ ]:
#pip install rfdetr

In [103]:
import supervision as sv
import cv2
from rfdetr import RFDETRMedium
from rfdetr.assets.coco_classes import COCO_CLASSES
from rfdetr import RFDETRSegMedium
import numpy as np
from court_line_detector import CourtLineDetector
import tempfile, os
from supervision import Position

In [54]:
#Load Models
model = RFDETRSegMedium()
court_line_detector = CourtLineDetector("keypoints_model.pth")

[2026-04-09 00:29:14] [INFO] rf-detr - File rf-detr-seg-medium.pt already exists with correct MD5 hash.


[2026-04-09 00:29:14] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-04-09 00:29:14] [WARNING] rf-detr - Using patch size 12 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-04-09 00:29:14] [INFO] rf-detr - File rf-detr-seg-medium.pt already exists with correct MD5 hash.


In [170]:
def get_player_indexes(detections, court_keypoints):
    # Extract keypoints for each group
    # Keypoint N → [court_keypoints[N*2], court_keypoints[N*2+1]]
    kp_group1 = np.array([[court_keypoints[k*2], court_keypoints[k*2+1]] for k in [8,9,12]])
    kp_group2 = np.array([[court_keypoints[k*2], court_keypoints[k*2+1]] for k in [10,11,13]])

    centers = detections.get_anchors_coordinates(Position.CENTER)
    class_ids = detections.class_id

    min_dist1, min_dist2 = float('inf'), float('inf')
    player1_idx, player2_idx = None, None

    for i, class_id in enumerate(class_ids):
        if class_id != 1:
            continue
        center = centers[i]

        # Distance from this detection to nearest keypoint in each group
        dist1 = np.min(np.linalg.norm(kp_group1 - center, axis=1))
        dist2 = np.min(np.linalg.norm(kp_group2 - center, axis=1))

        if dist1 < min_dist1:
            min_dist1 = dist1
            player1_idx = i

        if dist2 < min_dist2:
            min_dist2 = dist2
            player2_idx = i

    return {player1_idx, player2_idx}

In [182]:
def is_within_court(detection_index, detections, court_keypoints, margin=100):
    # Extract x,y for keypoints 0,1,2,3 from flat array [x0,y0,x1,y1,...]
    x0, y0 = court_keypoints[0], court_keypoints[1]  # top left
    x1, y1 = court_keypoints[2], court_keypoints[3]  # top right
    x2, y2 = court_keypoints[4], court_keypoints[5]  # bottom left
    x3, y3 = court_keypoints[6], court_keypoints[7]  # bottom right

    court_x_min = min(x0, x2)
    court_x_max = max(x1, x3)
    court_y_min = min(y0, y1)
    court_y_max = max(y2, y3)

    # Get center of the detection bounding box
    x1b, y1b, x2b, y2b = detections.xyxy[detection_index]
    cx = (x1b + x2b) / 2
    cy = (y1b + y2b) / 2

    return (court_x_min - margin <= cx <= court_x_max + margin and
            court_y_min - margin <= cy <= court_y_max + margin)

In [185]:
cap = cv2.VideoCapture("clip.mp4")
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# Set up output video writer
out = cv2.VideoWriter("output.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))

# Initialize annotators and detectors once (outside the loop)
court_line_detector = CourtLineDetector("keypoints_model.pth")
box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()

frame_idx = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break

    print(f"Processing frame {frame_idx + 1}/{total_frames}")

    # Save frame temporarily for model input
    temp_path = "temp_frame.jpg"
    cv2.imwrite(temp_path, frame)

    # Detect objects
    detections = model.predict(temp_path, threshold=0.1)
    detections.data = {k: v for k, v in detections.data.items() if isinstance(v, np.ndarray)}

    # Detect court lines
    court_keypoints = court_line_detector.predict(frame)

    # Filter objects
    all_indexes = []
    player_indexes = get_player_indexes(detections, court_keypoints)

    ball_indexes = [i for i, class_id in enumerate(detections.class_id) if class_id == 37]
    valid_ball_indexes = [i for i in ball_indexes if is_within_court(i, detections, court_keypoints)]

    for i, class_id in enumerate(detections.class_id):
        if i in player_indexes or i in valid_ball_indexes:
            all_indexes.append(i)
    filtered_detections = detections[all_indexes]

    # Annotate frame
    labels = [f"{COCO_CLASSES[class_id]}" for class_id in filtered_detections.class_id]
    annotated_frame = box_annotator.annotate(frame.copy(), filtered_detections)
    annotated_frame = label_annotator.annotate(annotated_frame, filtered_detections, labels)

    # Draw court lines
    annotated_frame = court_line_detector.draw_keypoints(annotated_frame, court_keypoints)

    # Write annotated frame to output video
    out.write(annotated_frame)
    frame_idx += 1

# Release resources
cap.release()
out.release()
print("Done! Saved to output.mp4")

Processing frame 1/238
Processing frame 2/238
Processing frame 3/238
Processing frame 4/238
Processing frame 5/238
Processing frame 6/238
Processing frame 7/238
Processing frame 8/238
Processing frame 9/238
Processing frame 10/238
Processing frame 11/238
Processing frame 12/238
Processing frame 13/238
Processing frame 14/238
Processing frame 15/238
Processing frame 16/238
Processing frame 17/238
Processing frame 18/238
Processing frame 19/238
Processing frame 20/238
Processing frame 21/238
Processing frame 22/238
Processing frame 23/238
Processing frame 24/238
Processing frame 25/238
Processing frame 26/238
Processing frame 27/238
Processing frame 28/238
Processing frame 29/238
Processing frame 30/238
Processing frame 31/238
Processing frame 32/238
Processing frame 33/238
Processing frame 34/238
Processing frame 35/238
Processing frame 36/238
Processing frame 37/238
Processing frame 38/238
Processing frame 39/238
Processing frame 40/238
Processing frame 41/238
Processing frame 42/238
P